# [HW05] Bead on Cone
We will be learning how to use python to facilitate calculations and study complex physical situations computationally.

**Step 1** Save a copy of this file in your PSU google-drive.

**Step 2** In your copy, hover your mouse over the [] brackets and a "play arrow" will appear - click it to run that bit of code.

**Step 3** if you see,

```
''' Write code here'''
```
written in <font color="orange">orange</font> - this is the place where you will have to add your own code using the variables explained in <font color="green">green</font>.

This activity guides you through numerically integrating the motion of a bead constrained to move on a cone.

🎯 **Assignment:**

* **(Section 0):** Equations of motion and setup
* **(Section A):** **Run code:** Run the provided code. Identify the adjustable constants used to explore the parameter space and record them.
* **(Section B):** **Plot motion and trajectory:** Write code to plot $r(t)$, $\omega(t)$, $l(t)$, and $\theta(t)$. Use a for-loop for the latter, then generate an $x$-$y$ plot for the bead's location.
* **(Section C):** **Vary parameters:** Systematically vary one of the parameters. Document your observations and generate a multiplot illustrating how the system behaves across these changes.
* **(Section D):** **Find stable orbit:** Determine if you can produce a stable trajectory where the bead never reaches the origin (for example, maintaining circular motion indefinitely). Provide supporting evidence, including a code snippet and dedicated plots for this scenario.

Document your findings/results in a separate submission file along with a copy of this notebook.



## Section 0: Equations of motion for bead on cone (theory)

Under the influence of gravity, a bead rolls down and around the surface of a cone.

**Variables:**
* $r$: Radial distance of the bead from the central axis
* $\theta$: Angular position in cylindrical coordinates
* $\omega$: Angular velocity of the bead
* $\alpha$: Angle of the cone relative to the vertical axis
* $g$: Acceleration due to gravity

---

**Equations of motion (EOM):**

1. Angular Momentum: $\frac{dl}{dt} = 0 $, where $l = m r^{2}\omega = m R_{0}^{2}\omega_{0}$ is a constant

2. Radial Motion: $\frac{d^{2}r}{dt^{2}} - r\omega^2\sin^{2}\alpha + g\sin\alpha \cos\alpha = 0$

We can revise these 2 equations of motion to 3 equations:
1. $\frac{d \theta}{dt}= \omega = \frac{R_{0}^{2}w_{0}}{r^{2}}$
2. $\frac{dv}{dt}=\frac{(R_{0}^{2}w_{0})^2}{r^3}\sin^{2}\alpha - g\sin\alpha \cos\alpha $
3. $\frac{dr}{dt} = v$ (from the definition)


Since our numerical solver `solve_ivp` (a `SciPy` function) requires a system of first-order ordinary differential equations (ODEs), we can convert the second-order radial equation into two first-order equations.

Let us define the state vector $ \mathbf{U}(t) = \left(\begin{array}{c} r(t) \\ v(t) \end{array}\right) $.

Taking a derivative and making some replacements, we obtain:

$$ \frac{d\mathbf{U}}{dt} = \left(\begin{array}{c} r'(t) \\ v'(t) \end{array}\right) = \left(\begin{array}{c} v(t) \\ \frac{(R_{0}^{2}w_{0})^2}{r^3}\sin^{2}\alpha - g\sin\alpha \cos\alpha \end{array}\right),$$

with initial conditions $ \mathbf{U}(0) = \left(\begin{array}{c} R_0 \\ v_0 \end{array}\right) $.


Once we solve for $r(t)$ and $v(t)$ using `solve_ivp`, we can subsequently compute the angular trajectory $\theta(t)$ using EOM (1).

## Section A: Bead on Code simulation

👉 Examine the code below to understand the algorithm, then run it. It should work as-is

In [ ]:
# (0) Import libraries
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

# (i) Derivative function
  # Inputs:
  #   'time': current time point
  #   'U': array w/ r = U[0] and v = U[1]
  # Output
  #   dU_dt = [r',v'] = [v,a]
  #   The function requires the current time point as an input during numerical integration.

def dU_dt(time, U):
  # Current velocity and acceleration
  r_prime=U[1]
  v_prime=((R0*R0*w0)**2/(U[0]**3)*np.sin(alpha)*np.sin(alpha)-g*np.sin(alpha)*np.cos(alpha))

  return [r_prime, v_prime]

# (ii) Definitions

# Constants and initial conditions
R0 = 4 # Initial radius
v0 = 0 # Initial radial velocity
w0 = 1.5 # Initial angular velocity in rad/s
alpha = 0.7 # Angle of cone w.r.t. vertical
g = 9.8
t_final = 4

# Time array and increment
time = np.linspace(0, t_final, 1000)  # Set up the array of time
dt=time[1]-time[0]

# State vector initial conditions U_0
U_0 = [R0, v0] # Initialize your U_0 matrix !  U_0 = [r(0)=R0, v(0)=v0]

# (iii) Numerical integration
U_pts = solve_ivp(dU_dt, (time[0],time[-1]),U_0, t_eval = time)

# (iv) Extract r(t)
r = U_pts.y[0,:]

# (v) Make it physically correct!
# Filter r and time to include only r > 0 values (keeping values in cone)
r_gt0 = r > 0.0
r_in_cone = r[r_gt0]
time_in_cone = time[r_gt0]

# (vi) Calculate omega using angular momentum conservation
omega = R0*R0*w0/r_in_cone**2

## Section B: Plot motion and trajectory

You do it: Now we have to look at the output and make sense of it.
1. Plot r(t) and omega(t)
2. Plot the angular momentum (l(t)) and make sure its a constant
3. Plot x(t) vs y(t)
4. write a few sentences about your plots and what they mean

### (B1) Time plots [$r(t)$, $\omega(t)$, $l(t)$]
Now, write code to:
1. Plot $r(t)$  and $\omega(t)$
2. Plot the angular momentum $ l(t) = mr^2 (t)\omega(t) $ and make sure it's constant

👉 Fill in the missing code wherever you see `'''Write code here'''`

In [ ]:
# (vii) Time plots
# Plot the radius [r_in_cone(time_in_cone)] and angular velocity [omega(time_in_cone)] v/s time
'''Write code here'''

# Plot the angular momentum (to verify that it's constant)
'''Write code here'''


### (B2) Trajectory of bead $x-y$

The goal is to plot the bead's position in the $x-y$ plane over time (without time on either axis). We will achieve this as follows:

1. **Find $\theta(t)$**: Calculate $\int \omega(t) dt$ with the Euler method
    ```
    for i in range(len(time_in_cone)-1)
      theta[i+1]=theta[i]+omega[i]*dt
    ```

2. **Convert to Cartesian:**
    * $y(t) = r(t) \sin(\theta)$
    * $x(t) = r(t) \cos(\theta)$

3. **Plot**

👉 Fill in the missing code wherever you see `'''Write code here'''`

In [ ]:
## (viii) Plot x v/s y ##

# Calculate the angle of the bead as it moves around the cone'''
theta_pts=np.zeros(len(time_in_cone))

# Create for-loop to calculate theta values
for i in range(len(time_in_cone)-1):
  theta_pts[i+1]=theta_pts[i]+omega[i]*dt

# Convert to Cartesian coordinates
'''Write code here'''
x=
y=
# Plot the x,y position of the bead
'''Write code here'''


## Section C: Your turn: Vary parameters

Select one parameter ($\alpha$, $\omega_0$, or $r_0$) to cycle through:

1. Create an array containing four values for your chosen parameter.
2. Write a for-loop to iterate over this array.
3. Copy the code from steps (ii)-(vi) and step (viii) inside the loop (noting that your chosen variable is now the array).
4. Modify step (viii) to display all four plots on one graph using `plt.subplot` [see PRE06].

In [ ]:
# Create array below
'''Write code here'''

# Create a figure and a 2x2 grid of subplots
'''Write code here'''

# Flatten the axes array for easier iteration
axes = axes.flatten()
count = 0

# Create for-loop to cycle through subplots
'''Write code here'''
   #copy the code from steps (ii)-(vi) and step (viii) inside the loop (noting that your chosen variable is now the array).
   '''Write code here'''


## Section D: Find stable orbit

Determine if you can produce a stable trajectory where the bead never reaches the origin (for example, maintaining circular motion indefinitely). Provide supporting evidence, including a code snippet and dedicated plots for this scenario. (Tip: You may need to increase t_final for a longer run time).

`Write answer here`

In [ ]:
'''Write code here'''

# Submission instructions

## Submitting as a **PDF**


To export as a PDF, in the toolbar navigate to File>>Print (or press CTRL+P), and then "print to PDF".

Submit the <font color="red">PDF</font> along with any other work (e.g. the paper copy of the recitation) to Canvas.
